In [1]:
from pathlib import Path
import sys

# Append the project root directory (Pulse_SLM) to Python's path
sys.path.append(str(Path.cwd().parent))

# Now import cleanly!
import torch
from src.model import TransformerDecoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
data_dir = Path("../data/processed")
data_path = data_dir / "processed_data.pt"
meta_path = data_dir / "pipeline_meta.pt"

In [3]:
if not data_path.exists() or not meta_path.exists():
    raise FileNotFoundError("Preprocessed artifacts not found. Run python -m src.data_preprocessing first.")
else:
    print("Data Found!")

Data Found!


In [4]:
import torch

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [6]:
data = torch.load(data_path, map_location=torch.device('cpu'))

In [7]:
type(data)

dict

In [8]:
data

{'X_train': tensor([[    0,     0,     0,  ...,     0,     0,     1],
         [    0,     0,     0,  ...,     0,     1,  3641],
         [    0,     0,     0,  ...,     1,  3641,  4046],
         ...,
         [    0,     0,     0,  ...,   365,    11,    26],
         [    0,     0,     0,  ...,    11,    26,  2145],
         [    0,     0,     0,  ...,    26,  2145, 19051]]),
 'y_train': tensor([ 3641,  4046,  1086,  ...,  2145, 19051,     2]),
 'X_val': tensor([[    0,     0,     0,  ...,     0,     0,     1],
         [    0,     0,     0,  ...,     0,     1, 13674],
         [    0,     0,     0,  ...,     1, 13674, 55791],
         ...,
         [    0,     0,     0,  ...,     1,   124,  2123],
         [    0,     0,     0,  ...,   124,  2123,   373],
         [    0,     0,     0,  ...,  2123,   373,  2123]]),
 'y_val': tensor([13674, 55791, 13674,  ...,   373,  2123,     2])}

In [9]:
X_train = data['X_train']
X_val = data['X_val']
y_train = data['y_train']
y_val = data['y_val']

X_train.shape, y_train.shape, X_val.shape, y_val.shape

(torch.Size([1864657, 567]),
 torch.Size([1864657]),
 torch.Size([194700, 567]),
 torch.Size([194700]))

In [10]:
import torch.nn as nn

In [11]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.param = nn.Parameter(torch.empty(self.vocab_size, self.d_model))
        nn.init.xavier_uniform_(self.param)
        
    def forward(self, x):
        return self.param[x]
        

In [12]:
emb = Embedding(vocab_size=X_train.shape[0], d_model=32).to(device)

In [13]:
# bx = next(iter(X_train))

In [14]:
# output = model(bx)

In [15]:
# output.shape

In [16]:
import math

In [17]:
class Positional_Embedding(nn.Module):
    def __init__(self, max_len,d_model):
        super().__init__()
        self.max_len = max_len 
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(d_model//2):
                denom = 10000**((2*i)/d_model)
                pe[pos, 2*i] = math.sin(pos/denom)
                pe[pos, 2*i+1] = math.cos(pos/denom)

        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :]    

In [18]:
class Multi_Head_Attention(nn.Module):
    def __init__(self, batch_size, num_heads, seq_len, head_dim ):
        super().__init__()
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.d_model = num_heads * head_dim
        self.W_q = nn.Parameter(torch.empty(self.d_model, self.d_model))
        self.W_k = nn.Parameter(torch.empty(self.d_model, self.d_model))
        self.W_v = nn.Parameter(torch.empty(self.d_model, self.d_model))
        self.b_q = nn.Parameter(torch.empty(1,self.d_model))
        self.b_k = nn.Parameter(torch.empty(1,self.d_model))
        self.b_v = nn.Parameter(torch.empty(1,self.d_model))
        nn.init.xavier_uniform_(self.W_k)
        nn.init.xavier_uniform_(self.W_q)
        nn.init.xavier_uniform_(self.W_v)
        nn.init.zeros_(self.b_k)
        nn.init.zeros_(self.b_q)
        nn.init.zeros_(self.b_v)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        # calculating Q, K and V
        q = x @ self.W_q + self.b_q
        k = x @ self.W_k + self.b_k
        v = x @ self.W_v + self.b_v

        # applying head 
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)

        # calcuating scaled dot product attention
        qk = q @ k.transpose(-2,-1) # calucating QK
        # finally scaled dot prodcut 
        scaled_qk = qk / math.sqrt(self.head_dim) 
        # applying mask
        scaled_qk = scaled_qk.masked_fill(mask, float('-inf'))
        
        # softmax
        scores = torch.softmax(scaled_qk, dim= -1)
        # context
        context = scores @ v

        output = context.transpose(1,2).contiguous().view(batch_size,seq_len,self.d_model)
        return output
        

In [19]:
class LayerNormalization(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps

        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        mean = x.mean(dim = -1, keepdim=True)
        var = x.var(dim = -1, keepdim = True, unbiased = False)
        x_norm = (x-mean) / torch.sqrt(var + self.eps)
        output = self.gamma * x_norm + self.beta

        return output
        

In [20]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, hidden_n):
        super().__init__()
        self.d_model = d_model
        self.hidden_n = hidden_n
        self.W_1 = nn.Parameter(torch.empty(self.d_model, self.hidden_n))
        self.b_1 = nn.Parameter((torch.empty(1, self.hidden_n)))
        self.W_2 = nn.Parameter(torch.empty(self.hidden_n, self.d_model))
        self.b_2 = nn.Parameter(torch.empty(1,self.d_model))
        nn.init.xavier_uniform_(self.W_1)
        nn.init.xavier_uniform_(self.W_2)
        nn.init.zeros_(self.b_1)
        nn.init.zeros_(self.b_2)


    def forward(self, x):
        Z_1 = x @ self.W_1 + self.b_1
        hidden = torch.relu(Z_1)
        Z_2 = hidden @ self.W_2 + self.b_2
        return Z_2

In [21]:
class TranformerDecoderBlock(nn.Module):
    def __init__(self, batch_size, num_heads, seq_len, head_dim, hidden_n):
        super().__init__()
        self.d_model = num_heads * head_dim

        self.mha = Multi_Head_Attention(
            batch_size=batch_size,
            num_heads = num_heads, 
            seq_len= seq_len,
            head_dim= head_dim
        )

        self.ln_1 = LayerNormalization(d_model=self.d_model)
        self.ln_2 = LayerNormalization(d_model=self.d_model)

        self.ffn = FeedForwardNetwork(d_model=self.d_model, hidden_n=hidden_n)

    def forward(self, x):
        mha_out = self.mha(self.ln_1(x))
        x1 = x + mha_out

        ffn_out = self.ffn(self.ln_2(x1))
        x2 = x1 + ffn_out

        return x2

In [22]:
decoder = TranformerDecoderBlock(
    batch_size=32,
    num_heads=4,
    seq_len=567,
    head_dim = 8,
    hidden_n=128
).to(device)

In [23]:

batch_x = X_train[:32].to(device)  # shape: (32, 567)

x_emb = emb(batch_x)  # shape: (32, 567, 32)

pos_emb = Positional_Embedding(max_len=567, d_model=32).to(device)
x_in = pos_emb(x_emb)

out = decoder(x_in)

print("Output shape:", out.shape)

Output shape: torch.Size([32, 567, 32])


In [24]:
out

tensor([[[-1.0614e+00,  7.5288e-01,  1.3870e-01,  ...,  6.4075e-01,
          -1.0464e+00,  4.2558e-01],
         [ 8.2688e-04,  5.7226e-01,  9.3184e-01,  ...,  1.0248e+00,
          -1.0310e+00,  4.0307e-01],
         [ 3.5853e-01, -2.3163e-01,  1.2040e+00,  ...,  1.1589e+00,
          -8.1644e-01,  3.7804e-01],
         ...,
         [-8.7674e-01,  1.8095e+00, -4.2516e-01,  ..., -4.0264e-01,
           4.5009e-02,  1.7919e+00],
         [-1.0613e-01,  2.5904e+00, -8.6024e-01,  ..., -4.8142e-01,
          -7.4073e-02,  1.9444e+00],
         [ 1.0014e+00,  2.4319e+00, -1.1764e+00,  ..., -4.5504e-01,
          -5.0116e-02,  2.0186e+00]],

        [[-1.0614e+00,  7.5288e-01,  1.3870e-01,  ...,  6.4075e-01,
          -1.0464e+00,  4.2558e-01],
         [ 8.2688e-04,  5.7226e-01,  9.3184e-01,  ...,  1.0248e+00,
          -1.0310e+00,  4.0307e-01],
         [ 3.5853e-01, -2.3163e-01,  1.2040e+00,  ...,  1.1589e+00,
          -8.1644e-01,  3.7804e-01],
         ...,
         [-8.7674e-01,  1

In [25]:
import torch
from src.model import TransformerDecoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the full modular model
model = TransformerDecoder(
    vocab_size=60000,
    d_model=32,
    num_layers=4,
    num_heads=4,
    seq_len=567,
    head_dim=8,
    hidden_n=128,
).to(device)

# Forward pass test with your batch
batch_x = X_train[:32].to(device)
logits = model(batch_x)

print("Modular model forward pass successful!")
print("Logits shape:", logits.shape)  # Expected: (32, 567, 60000)

Modular model forward pass successful!
Logits shape: torch.Size([32, 567, 60000])
